# Week 1: Setup and Data Loading

> **Milestone:** Setup environment, load data
> **Deliverable:** Notebook 01 complete

*From PROJECT_BRIEF.md - Phase 1: Foundation (Weeks 1-4)*

In [ ]:
# Import libraries
import geopandas as gpd
import rasterio
from rasterio.plot import show
import pandas as pd
import numpy as np
import os

# Set working directory
wd = '/home/recursivex/my_projects/my_career/2026_roadmap_fully_funded_opportunities/hydrogen_storage_site_selection'
os.chdir(wd)

# Verify folder structure
print(f'Working directory: {wd}')
print('\nFolder structure:')
for root, dirs, files in os.walk('.'):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace('.').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:
        print(f'{subindent}{file}')
    if len(files) > 5:
        print(f'{subindent}... and {len(files)-5} more')

## Objective: Load and explore data layers

We will load the 5 data layers specified in the project:
- Solar Irradiance (Global Solar Atlas) - GeoTIFF
- Water Stress (WRI Aqueduct) - Raster/Shapefile
- Subsurface Geology (USGS Basins / Depleted Fields) - Shapefile
- End-User Proximity - Shapefile
- Population Density - Raster

## Step 1: Check for existing data

*If you have downloaded data already, place it in `data/raw/` directory.*
*Otherwise, we will proceed with placeholder descriptions and create
synthetic data for exploration.*

## Step 2: Create Synthetic Data (if no raw data available)

*Since you may not have downloaded the actual datasets yet, we'll create
synthetic data that mimics the structure of the real data layers. This
allows us to build and test the framework now, and you can replace the
data later with real datasets.*

***For your real project:** Download these datasets:
*- Global Solar Atlas: https://globalsolaratlas.info
*- WRI Aqueduct Water Stress: https://www.wri.org/aqueduct
*- USGS Basins: https://pubs.usgs.gov
*- SEDAC Population Density: https://sedac.ciesin.columbia.edu

***What you'll get:** GeoTIFF and shapefiles clipped to your region of
interest (Southern Africa - e.g., Namibia, Botswana, South Africa west coast)*

In [ ]:
# Create synthetic solar irradiance GeoTIFF (simulates Global Solar Atlas data)
import rasterio
from rasterio.transform import Affine
from rasterio.crs import CRS

# Synthetic solar irradiance data for Southern Africa region
height, width = 100, 100
solar_data = np.random.uniform(1500, 3000, (height, width))  # W/m2 range for Africa

# Create transform for a region in Southern Africa (approx. Namibia/Botswana area)
transform = Affine(0.1, 0, 12, 0, -0.1, -10)

# Create CRS (WGS84)
crs = CRS.from_epsg(4326)

# Write GeoTIFF
with rasterio.open(
    'data/raw/solar_irradiance.tif',
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype='float32',
    crs=crs,
    transform=transform,
) as dst:
    dst.write(solar_data, 1)

print('Created synthetic solar_irradiance.tif')

# Create synthetic water stress raster
water_data = np.random.uniform(0, 1, (height, width))  # 0=low stress, 1=high stress

with rasterio.open(
    'data/raw/water_stress.tif',
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype='float32',
    crs=crs,
    transform=transform,
) as dst:
    dst.write(water_data, 1)

print('Created synthetic water_stress.tif')

# Create synthetic population density raster
pop_data = np.random.uniform(10, 500, (height, width))  # people per km2

with rasterio.open(
    'data/raw/population_density.tif',
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=1,
    dtype='float32',
    crs=crs,
    transform=transform,
) as dst:
    dst.write(pop_data, 1)

print('Created synthetic population_density.tif')

# Create synthetic geology shapefile (depleted oil/gas fields)
import geopandas as gpd
from shapely.geometry import Point, Polygon

# Create sample points representing geological basins
points = [
    Point(15.0, -15.0),  # Basin 1
    Point(18.0, -18.0),  # Basin 2
    Point(20.0, -15.0),  # Basin 3
    Point(22.0, -13.0),  # Basin 4
    Point(25.0, -13.0),  # Basin 5
]

# Create polygon buffer around each point
geometries = []
for i, p in enumerate(points):
    polygon = p.buffer(2.0)  # 2-degree buffer
    geometries.append(polygon)

# Create GeoDataFrame
geology_gdf = gpd.GeoDataFrame({
    'basin_id': range(1, len(geometries) + 1),
    'rock_type': ['Sandstone', 'Limestone', 'Sandstone', 'Shale', 'Limestone'],
    'porosity': [0.25, 0.35, 0.28, 0.15, 0.30],
    'permeability': [150, 200, 180, 80, 220],
}, geometry=geometries)

geology_gdf = geology_gdf.set_crs(epsg=4326, inplace=False)
geology_gdf.to_file('data/raw/usgs_basins.shp')
print('Created synthetic usgs_basins.shp')

# Create synthetic end-user proximity shapefile (ports/industrial centers)
user_points = [
    Point(18.0, -14.0),  # Port city
    Point(22.0, -16.0),  # Industrial center
    Point(24.0, -14.5),  # Another industrial area
]

user_geometries = [p.buffer(1.0) for p in user_points]

user_gdf = gpd.GeoDataFrame({
    'user_id': range(1, len(user_geometries) + 1),
    'facility_type': ['Port', 'Industrial', 'Port'],
    'capacity_mw': [500, 300, 450],
}, geometry=user_geometries)

user_gdf = user_gdf.set_crs(epsg=4326, inplace=False)
user_gdf.to_file('data/raw/end_users.shp')
print('Created synthetic end_users.shp')

print('\nAll synthetic data created successfully in data/raw/')
print('Files:', os.listdir('data/raw'))